# 14 Pretraining_Data_And_Objective

## Problem

模型结构之外，预训练到底在学什么？next-token prediction 和 cross entropy loss 是怎样把语言规律、知识共现和模式压进参数里的？

## Dependencies

需要基本理解 logits、softmax、token id、context length。


## Module_Goals

- 理解 next-token prediction 的训练目标
- 理解 logits、概率分布、loss 之间的关系
- 理解 batch、context length、训练 token 数的基本概念
- 理解预训练阶段在整个多阶段训练链路中的职责

## Scope_And_Approach

这个 notebook 不只是解释一个 loss 公式，而是说明 pretraining 为什么是整个系统的底座。重点是把“预训练在训练主线里到底负责什么”讲清楚，再用最小例子落到 cross entropy 和 token-level objective。


## Context_And_Motivation

预训练可以先简单理解成一个超大规模的填空过程：

- 给模型一段前文。
- 让它预测下一个 token。
- 如果猜得不好，就调整参数。
- 在海量语料上反复这么做。

这个目标看上去很简单，但它逼着模型学到大量结构：语法、常识、代码模式、文风、知识共现、推理模板。因为想把“下一个 token”猜准，本身就要求模型尽可能把上下文里的规律压进参数里。

从多阶段训练视角看，pretraining 的职责不是把模型调成一个具体助手，而是先让模型拥有足够强的通用生成和表示能力。没有这一层，后面的 continued training、SFT、RLHF 都没有稳定底座。


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 假设词表大小为 5，当前要预测的正确 token 是 id=2
logits = np.array([1.2, 0.3, 2.4, -0.5, 0.1])
target_id = 2

def softmax(logits):
    shifted = logits - np.max(logits)
    exp = np.exp(shifted)
    return exp / np.sum(exp)

probs = softmax(logits)
loss = -np.log(probs[target_id])

print('logits =', logits)
print('probs =', probs)
print('target_prob =', probs[target_id])
print('cross_entropy_loss =', loss)


In [ ]:
# 一个最小 batch 版本
batch_logits = np.array([
    [2.0, 0.5, 0.2],
    [0.1, 1.5, 0.3],
    [0.2, 0.4, 1.8],
])
batch_targets = np.array([0, 1, 2])
batch_probs = np.apply_along_axis(softmax, 1, batch_logits)
batch_losses = -np.log(batch_probs[np.arange(len(batch_targets)), batch_targets])

print('batch_probs =')
print(batch_probs)
print('batch_losses =', batch_losses)
print('mean_loss =', batch_losses.mean())


In [ ]:
# 一个最小训练阶段定位示意
training_stages = {
    'pretraining': 'learn broad language and world patterns',
    'continued_training': 'shift capacity toward domain-critical skills',
    'sft': 'shape instruction-following behavior',
    'rlhf': 'shape preference-aligned behavior',
}

for stage, role in training_stages.items():
    print(f'{stage}: {role}')


## Math

预训练的核心 token-level objective 可以写成：

$$L = -\log p(x_t \mid x_{<t})$$

对一整段序列来说，就是把所有位置上的 next-token loss 加总或取平均。这个目标本身很朴素，但只要上下文足够长、语料足够广、优化足够稳定，模型就会从中学出大量结构信息。

## Trade_Offs_And_Risk_Points

- 把预训练理解成背答案。实际上它是在海量上下文约束下学习统计结构。
- 只盯着 loss 数值，不理解它本质上是在惩罚“给正确 token 的概率不够高”。
- 把 token 数、batch size、context length 混成一个概念。它们都影响训练成本，但含义不同。
- 误以为预训练结束就已经得到最终产品。预训练更像是在搭地基，而不是做最后成品。

## Checkpoints

- 把 target 改成模型不太相信的 token，观察 loss 如何变大。
- 思考为什么预训练语料规模和多样性会强烈影响模型能力。
- 解释为什么 next-token prediction 虽然简单，却足以支持复杂能力涌现。
- 用自己的话说明 pretraining 在多阶段训练链路里的职责。

## Summary

预训练阶段做的事并不花哨，但它是整个大模型能力的底座。它主要负责建立广义语言和知识表示，而不是直接塑造成具体助手。后面的 continued training、SFT、RLHF 都是在这个底座之上继续塑形。

## Next_Module

下一步进入 SFT，也就是模型从“会续写”进一步转向“更会按指令回答”的阶段。
